In [1]:
# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# to perform PCA
from sklearn.decomposition import PCA

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)

# to scale the data using z-score
from sklearn.preprocessing import StandardScaler

# to compute distances
from scipy.spatial.distance import cdist

# to perform k-means clustering and compute silhouette scores
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# to visualize the elbow curve and silhouette scores
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer


In [2]:
df1 = pd.read_csv('studentVle1.csv')

In [3]:
df2 = pd.read_csv('student_registration_cleaned.csv')

In [4]:
# Import all CSV files
assessment_df = pd.read_csv('assessment_df.csv')
courses_cleaned_merged = pd.read_csv('courses_cleaned_merged.csv')
student_data_cleaned = pd.read_csv('student_data_cleaned.csv')
studentAssessment_df = pd.read_csv('studentAssessment_df.csv')
studentVle = pd.read_csv('studentVle1.csv')
vle_df = pd.read_csv('vle.csv')

In [5]:
assessment_df.info()
print("\nFirst few rows:")
print(assessment_df.head())
print("\nShape:", assessment_df.shape)
print("\nBasic statistics:")
print(assessment_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_assessment             206 non-null    int64  
 1   assessment_type           206 non-null    object 
 2   date                      206 non-null    float64
 3   weight                    206 non-null    float64
 4   code_module_presentation  206 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 8.2+ KB

First few rows:
   id_assessment assessment_type   date  weight code_module_presentation
0           1752             tma   19.0    10.0                aaa-2013j
1           1753             tma   54.0    20.0                aaa-2013j
2           1754             tma  117.0    20.0                aaa-2013j
3           1755             tma  166.0    20.0                aaa-2013j
4           1756             tma  215.0    30.0                aaa-

In [6]:
# Check column names in each dataframe to determine merge keys
print("assessment_df columns:", assessment_df.columns.tolist())
print("\ncourses_cleaned_merged columns:", courses_cleaned_merged.columns.tolist())
print("\nstudent_data_cleaned columns:", student_data_cleaned.columns.tolist())
print("\nstudentAssessment_df columns:", studentAssessment_df.columns.tolist())
print("\nvle_df columns:", vle_df.columns.tolist())
print("\nstudentVle columns:", studentVle.columns.tolist())

assessment_df columns: ['id_assessment', 'assessment_type', 'date', 'weight', 'code_module_presentation']

courses_cleaned_merged columns: ['module_presentation_length', 'code_module_presentation']

student_data_cleaned columns: ['imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_presentation', 'code_module_student_presentation', 'gender_highest_edu_region', 'studied_credits_finalresult']

studentAssessment_df columns: ['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

vle_df columns: ['id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']

studentVle columns: ['id_site', 'date', 'sum_click', 'code_module_site_presentation', 'code_module_student_presentation']


In [7]:
# Step 1: Start with studentAssessment_df as the base
master_df = studentAssessment_df.copy()
print("Step 1 - Starting with studentAssessment_df")
print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}\n")

Step 1 - Starting with studentAssessment_df
Shape: (173912, 6)
Columns: ['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']



In [8]:
# Step 2: Merge with assessment_df using id_assessment
master_df = master_df.merge(assessment_df, on='id_assessment', how='left')
print("Step 2 - Merged with assessment_df on id_assessment")
print(f"Shape: {master_df.shape}")
print(f"New columns added: {['assessment_type', 'date', 'weight', 'code_module_presentation']}\n")

Step 2 - Merged with assessment_df on id_assessment
Shape: (173912, 10)
New columns added: ['assessment_type', 'date', 'weight', 'code_module_presentation']



In [9]:
# Step 3: Merge with student_data_cleaned using code_module_presentation and id_student
master_df = master_df.merge(student_data_cleaned, on='code_module_presentation', how='left')
print("Step 3 - Merged with student_data_cleaned on code_module_presentation")
print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}\n")

Step 3 - Merged with student_data_cleaned on code_module_presentation
Shape: (173912, 18)
Columns: ['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'date', 'weight', 'code_module_presentation', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_student_presentation', 'gender_highest_edu_region', 'studied_credits_finalresult']



In [10]:
# Step 4: Merge with courses_cleaned_merged using code_module_presentation
master_df = master_df.merge(courses_cleaned_merged, on='code_module_presentation', how='left')
print("Step 4 - Merged with courses_cleaned_merged on code_module_presentation")
print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}\n")

Step 4 - Merged with courses_cleaned_merged on code_module_presentation
Shape: (173912, 19)
Columns: ['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'date', 'weight', 'code_module_presentation', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_student_presentation', 'gender_highest_edu_region', 'studied_credits_finalresult', 'module_presentation_length']



In [11]:
# Step 5: Merge with vle_df
# First, we need to rename columns for matching: code_module_presentation needs to split or match code_module and code_presentation
# For now, we'll do a left merge, but this may need adjustment based on the data structure
master_df = master_df.merge(vle_df, left_on='code_module_presentation', right_on='code_module', how='left')
print("Step 5 - Merged with vle_df")
print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}\n")

Step 5 - Merged with vle_df
Shape: (173912, 25)
Columns: ['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'date', 'weight', 'code_module_presentation', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_student_presentation', 'gender_highest_edu_region', 'studied_credits_finalresult', 'module_presentation_length', 'id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']



In [12]:
# Step 6: Merge with studentVle using code_module_student_presentation and id_site (or other matching keys)
master_df = master_df.merge(studentVle, on=['id_site', 'code_module_student_presentation'], how='left')
print("Step 6 - Merged with studentVle on id_site and code_module_student_presentation")
print(f"Final Master DataFrame Shape: {master_df.shape}")
print(f"\nFinal Columns ({len(master_df.columns)}): {master_df.columns.tolist()}")
print(f"\nMaster DataFrame Info:")
print(master_df.info())

Step 6 - Merged with studentVle on id_site and code_module_student_presentation
Final Master DataFrame Shape: (173912, 28)

Final Columns (28): ['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'date_x', 'weight', 'code_module_presentation', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_student_presentation', 'gender_highest_edu_region', 'studied_credits_finalresult', 'module_presentation_length', 'id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to', 'date_y', 'sum_click', 'code_module_site_presentation']

Master DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 28 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Unnamed: 0                        173912 non-null  int64  
 1   id_assessment       

In [13]:
# Check studentVle data
print("StudentVle Original Data:")
print(f"Shape: {studentVle.shape}")
print(f"Columns: {studentVle.columns.tolist()}")
print(f"\nFirst few rows:")
print(studentVle.head(10))
print(f"\nData types:")
print(studentVle.dtypes)
print(f"\nMissing values:")
print(studentVle.isnull().sum())
print(f"\nBasic statistics:")
print(studentVle.describe())

StudentVle Original Data:
Shape: (10655280, 5)
Columns: ['id_site', 'date', 'sum_click', 'code_module_site_presentation', 'code_module_student_presentation']

First few rows:
   id_site  date  sum_click code_module_site_presentation  \
0   546652   -10          4              aaa-2013j-546652   
1   546652   -10          1              aaa-2013j-546652   
2   546652   -10          1              aaa-2013j-546652   
3   546614   -10         11              aaa-2013j-546614   
4   546714   -10          1              aaa-2013j-546714   
5   546652   -10          8              aaa-2013j-546652   
6   546876   -10          2              aaa-2013j-546876   
7   546688   -10         15              aaa-2013j-546688   
8   546662   -10         17              aaa-2013j-546662   
9   546890   -10          1              aaa-2013j-546890   

  code_module_student_presentation  
0                  aaa-2013j-28400  
1                  aaa-2013j-28400  
2                  aaa-2013j-28400  
3    

In [14]:
# Step 1: Aggregate studentVle data by id_site, code_module_student_presentation, and date
# This will sum the clicks and get count of activities
studentVle_agg = studentVle.groupby(['id_site', 'code_module_student_presentation', 'date']).agg({
    'sum_click': ['sum', 'mean', 'count'],
    'code_module_site_presentation': 'first'
}).reset_index()

# Flatten column names
studentVle_agg.columns = ['id_site', 'code_module_student_presentation', 'date', 
                          'total_clicks', 'avg_clicks', 'activity_count', 'code_module_site_presentation']

print("Aggregated StudentVle Data:")
print(f"Shape: {studentVle_agg.shape}")
print(f"Columns: {studentVle_agg.columns.tolist()}")
print(f"\nFirst few rows:")
print(studentVle_agg.head())
print(f"\nBasic statistics:")
print(studentVle_agg[['total_clicks', 'avg_clicks', 'activity_count']].describe())

Aggregated StudentVle Data:
Shape: (8459320, 7)
Columns: ['id_site', 'code_module_student_presentation', 'date', 'total_clicks', 'avg_clicks', 'activity_count', 'code_module_site_presentation']

First few rows:
   id_site code_module_student_presentation  date  total_clicks  avg_clicks  \
0   526721                 fff-2013b-101306   -18            13        13.0   
1   526721                 fff-2013b-101306    -2            11        11.0   
2   526721                 fff-2013b-101306    -1             1         1.0   
3   526721                 fff-2013b-101306     3             1         1.0   
4   526721                 fff-2013b-101306     8             4         4.0   

   activity_count code_module_site_presentation  
0               1              fff-2013b-526721  
1               1              fff-2013b-526721  
2               1              fff-2013b-526721  
3               1              fff-2013b-526721  
4               1              fff-2013b-526721  

Basic statist

In [15]:
# Step 2: Merge aggregated studentVle back to master_df
# Using the same keys as before: id_site, code_module_student_presentation, and the VLE date key
print("Before merge - Master DF shape:", master_df.shape)
print("Missing values before merge:")
print(f"id_site: {master_df['id_site'].isna().sum()}")
print(f"code_module_student_presentation: {master_df['code_module_student_presentation'].isna().sum()}")

# Merge aggregated studentVle with master_df using the existing date_y column
master_df = master_df.merge(
    studentVle_agg[['id_site', 'code_module_student_presentation', 'date',
                     'total_clicks', 'avg_clicks', 'activity_count']],
    left_on=['id_site', 'code_module_student_presentation', 'date_y'],
    right_on=['id_site', 'code_module_student_presentation', 'date'],
    how='left'
)

print(f"\nAfter merge - Master DF shape: {master_df.shape}")
print(f"New columns added: {['total_clicks', 'avg_clicks', 'activity_count']}")

Before merge - Master DF shape: (173912, 28)
Missing values before merge:
id_site: 173912
code_module_student_presentation: 173912

After merge - Master DF shape: (173912, 32)
New columns added: ['total_clicks', 'avg_clicks', 'activity_count']


In [16]:
# Check master_df columns to understand the date structure
print("Master DF columns:")
print(master_df.columns.tolist())
print("\nColumns with 'date':")
print([col for col in master_df.columns if 'date' in col.lower()])

print("\nIf the merge output is correct, you can proceed to missing-value checks.")

Master DF columns:
['Unnamed: 0', 'id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'date_x', 'weight', 'code_module_presentation', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_student_presentation', 'gender_highest_edu_region', 'studied_credits_finalresult', 'module_presentation_length', 'id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to', 'date_y', 'sum_click', 'code_module_site_presentation', 'date', 'total_clicks', 'avg_clicks', 'activity_count']

Columns with 'date':
['date_submitted', 'date_x', 'date_y', 'date']

If the merge output is correct, you can proceed to missing-value checks.


In [17]:
# Step 3: Check missing values before filling
print("Missing values before filling:")
print(master_df.isnull().sum())
print(f"\nTotal missing values: {master_df.isnull().sum().sum()}")
print(f"\nPercentage of missing values per column:")
print((master_df.isnull().sum() / len(master_df) * 100).round(2))

Missing values before filling:
Unnamed: 0                               0
id_assessment                            0
id_student                               0
date_submitted                           0
is_banked                                0
score                                    0
assessment_type                          0
date_x                                   0
weight                                   0
code_module_presentation                 0
imd_band                            173912
age_band                            173912
num_of_prev_attempts                173912
studied_credits                     173912
disability                          173912
code_module_student_presentation    173912
gender_highest_edu_region           173912
studied_credits_finalresult         173912
module_presentation_length               0
id_site                             173912
code_module                         173912
code_presentation                   173912
activity_type          

In [18]:
# Step 4: Fill missing values with appropriate strategies
# Separate numeric and categorical columns with missing values

# For numeric columns: fill with 0 (since they represent counts/metrics)
numeric_cols_to_fill = ['num_of_prev_attempts', 'studied_credits', 'week_from', 'week_to', 
                        'date_y', 'sum_click', 'total_clicks', 'avg_clicks', 'activity_count']

# For categorical columns: fill with 'Unknown' or 'Not Available'
categorical_cols_to_fill = ['imd_band', 'age_band', 'disability', 'gender_highest_edu_region',
                            'studied_credits_finalresult', 'code_module_student_presentation',
                            'id_site', 'code_module', 'code_presentation', 'activity_type',
                            'code_module_site_presentation', 'date']

print("Filling missing values...")
print(f"Numeric columns to fill: {numeric_cols_to_fill}")
print(f"Categorical columns to fill: {categorical_cols_to_fill}")

# Fill numeric columns with 0
for col in numeric_cols_to_fill:
    if col in master_df.columns:
        master_df[col] = master_df[col].fillna(0)

# Fill categorical columns with 'Unknown'
for col in categorical_cols_to_fill:
    if col in master_df.columns:
        master_df[col] = master_df[col].fillna('Unknown')

print(f"\nMissing values after filling:")
print(master_df.isnull().sum().sum())
print(f"\nMaster DataFrame final shape: {master_df.shape}")
print(f"\nSample of filled data:")
print(master_df.head(10))

Filling missing values...
Numeric columns to fill: ['num_of_prev_attempts', 'studied_credits', 'week_from', 'week_to', 'date_y', 'sum_click', 'total_clicks', 'avg_clicks', 'activity_count']
Categorical columns to fill: ['imd_band', 'age_band', 'disability', 'gender_highest_edu_region', 'studied_credits_finalresult', 'code_module_student_presentation', 'id_site', 'code_module', 'code_presentation', 'activity_type', 'code_module_site_presentation', 'date']

Missing values after filling:
0

Master DataFrame final shape: (173912, 32)

Sample of filled data:
   Unnamed: 0  id_assessment  id_student  date_submitted  is_banked  score  \
0           0           1752       11391              18          0   78.0   
1           1           1752       28400              22          0   70.0   
2           2           1752       31604              17          0   72.0   
3           3           1752       32885              26          0   69.0   
4           4           1752       38053          

In [19]:
# Final Summary
print("=" * 80)
print("FINAL MASTER DATAFRAME SUMMARY")
print("=" * 80)
print(f"\n✓ Final shape: {master_df.shape} (rows, columns)")
print(f"✓ Total missing values: {master_df.isnull().sum().sum()}")
print(f"✓ Data types:")
print(f"  - Numeric (float64, int64): {(master_df.dtypes.isin(['float64', 'int64'])).sum()}")
print(f"  - Object (string): {(master_df.dtypes == 'object').sum()}")
print(f"\n✓ Column list ({len(master_df.columns)} columns):")
for i, col in enumerate(master_df.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\n✓ Data info:")
print(f"  - Memory usage: {master_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"\n✓ Aggregated StudentVle metrics added:")
print(f"  - total_clicks: Total clicks per student/site/date")
print(f"  - avg_clicks: Average clicks per activity")
print(f"  - activity_count: Number of activities per student/site/date")

FINAL MASTER DATAFRAME SUMMARY

✓ Final shape: (173912, 32) (rows, columns)
✓ Total missing values: 0
✓ Data types:
  - Numeric (float64, int64): 0
  - Object (string): 14

✓ Column list (32 columns):
   1. Unnamed: 0
   2. id_assessment
   3. id_student
   4. date_submitted
   5. is_banked
   6. score
   7. assessment_type
   8. date_x
   9. weight
  10. code_module_presentation
  11. imd_band
  12. age_band
  13. num_of_prev_attempts
  14. studied_credits
  15. disability
  16. code_module_student_presentation
  17. gender_highest_edu_region
  18. studied_credits_finalresult
  19. module_presentation_length
  20. id_site
  21. code_module
  22. code_presentation
  23. activity_type
  24. week_from
  25. week_to
  26. date_y
  27. sum_click
  28. code_module_site_presentation
  29. date
  30. total_clicks
  31. avg_clicks
  32. activity_count

✓ Data info:
  - Memory usage: 172.16 MB

✓ Aggregated StudentVle metrics added:
  - total_clicks: Total clicks per student/site/date
  - avg_c

In [20]:
master_df.shape

(173912, 32)

In [21]:
# Inspect course duration and final result values for 25% completion logic
print('Unique final result values:')
print(student_data_cleaned['studied_credits_finalresult'].value_counts(dropna=False))

print('\nVLE module duration summary:')
print(vle_df.groupby(['code_module', 'code_presentation'])['week_to'].max().reset_index().head(20))

print('\nSample of master_df date columns:')
print(master_df[['date_x', 'date_y']].head())
print('\nCount of date_y missing values:', master_df['date_y'].isna().sum())


Unique final result values:
studied_credits_finalresult
60_pass            6961
60_withdrawn       4571
60_fail            3677
120_withdrawn      2549
120_pass           2026
30_pass            1580
60_distinction     1542
120_fail           1238
90_withdrawn       1145
90_pass            1054
30_fail            1011
90_fail             649
30_withdrawn        628
30_distinction      530
120_distinction     515
180_withdrawn       438
150_withdrawn       344
90_distinction      296
150_pass            236
180_pass            200
180_fail            160
150_fail            151
240_withdrawn       150
210_withdrawn       101
75_pass              44
70_pass              42
150_distinction      38
210_pass             37
240_pass             36
75_withdrawn         33
180_distinction      32
240_fail             26
270_withdrawn        25
210_fail             25
70_withdrawn         23
75_fail              23
45_pass              18
70_fail              18
130_pass             16
300_with

In [22]:
# Inspect whether master_df has key columns for studentVle merge
print('master_df code_module_student_presentation non-null:', master_df['code_module_student_presentation'].notna().sum())
print('master_df id_site non-null:', 'id_site' in master_df.columns and master_df['id_site'].notna().sum())
print('\nstudent_data_cleaned sample columns:')
print(student_data_cleaned[['code_module_presentation', 'code_module_student_presentation']].head())
print('\nUnique code_module_presentation in master_df:', master_df['code_module_presentation'].nunique())
print('Unique code_module_presentation in student_data_cleaned:', student_data_cleaned['code_module_presentation'].nunique())
print('\nFirst 5 master_df code_module_presentation values:')
print(master_df['code_module_presentation'].unique()[:10])
print('\nFirst 5 student_data_cleaned code_module_presentation values:')
print(student_data_cleaned['code_module_presentation'].unique()[:10])

master_df code_module_student_presentation non-null: 173912
master_df id_site non-null: 173912

student_data_cleaned sample columns:
  code_module_presentation code_module_student_presentation
0                aaa_2013j                  aaa_2013j_11391
1                aaa_2013j                  aaa_2013j_28400
2                aaa_2013j                  aaa_2013j_30268
3                aaa_2013j                  aaa_2013j_31604
4                aaa_2013j                  aaa_2013j_32885

Unique code_module_presentation in master_df: 22
Unique code_module_presentation in student_data_cleaned: 22

First 5 master_df code_module_presentation values:
['aaa-2013j' 'aaa-2014j' 'bbb-2013b' 'bbb-2013j' 'bbb-2014b' 'bbb-2014j'
 'ccc-2014b' 'ccc-2014j' 'ddd-2013b' 'ddd-2013j']

First 5 student_data_cleaned code_module_presentation values:
['aaa_2013j' 'aaa_2014j' 'bbb_2013b' 'bbb_2013j' 'bbb_2014b' 'bbb_2014j'
 'ccc_2014b' 'ccc_2014j' 'ddd_2013b' 'ddd_2013j']


In [23]:
# Adjust studentVle for 25% completion and create a dropout-risk dataset

# Normalize student-level keys in student_data_cleaned
student_data_cleaned = student_data_cleaned.copy()
student_data_cleaned['code_module_presentation_norm'] = student_data_cleaned['code_module_presentation'].str.replace('_', '-', regex=False)
student_data_cleaned['code_module_student_presentation_norm'] = student_data_cleaned['code_module_student_presentation'].str.replace('_', '-', regex=False)
student_data_cleaned['id_student'] = student_data_cleaned['code_module_student_presentation'].str.rsplit(pat='_', n=1).str[-1].astype(int)
student_data_cleaned['dropout_flag'] = student_data_cleaned['studied_credits_finalresult'].str.contains('withdrawn', case=False, na=False).astype(int)

# Build a new student-level master frame that can join VLE features
master_dropout_df = studentAssessment_df.merge(assessment_df, on='id_assessment', how='left')
master_dropout_df = master_dropout_df.merge(
    student_data_cleaned[
        ['id_student', 'code_module_presentation_norm', 'code_module_student_presentation_norm',
         'dropout_flag', 'studied_credits_finalresult', 'imd_band', 'age_band',
         'num_of_prev_attempts', 'studied_credits', 'disability', 'gender_highest_edu_region']
    ],
    left_on=['id_student', 'code_module_presentation'],
    right_on=['id_student', 'code_module_presentation_norm'],
    how='left'
)

# Prepare course length thresholds for 25% completion
course_length = (
    vle_df
    .dropna(subset=['week_to'])
    .assign(
        code_module_lower=lambda df: df['code_module'].str.lower(),
        code_presentation_lower=lambda df: df['code_presentation'].str.lower()
    )
    .groupby(['code_module_lower', 'code_presentation_lower'], as_index=False)
    ['week_to']
    .max()
    .rename(columns={
        'code_module_lower': 'code_module',
        'code_presentation_lower': 'code_presentation',
        'week_to': 'course_weeks'
    })
)
course_length['module_presentation'] = course_length['code_module'] + '-' + course_length['code_presentation']
course_length['threshold_days'] = np.ceil(course_length['course_weeks'] * 7 * 0.25)

# Normalize studentVle keys and attach module presentation info
studentVle = studentVle.copy()
studentVle['code_module_student_presentation_norm'] = studentVle['code_module_student_presentation'].str.replace('_', '-', regex=False)
studentVle['module_presentation'] = studentVle['code_module_student_presentation_norm'].str.rsplit(pat='-', n=1).str[0]

studentVle_25pct = studentVle.merge(
    course_length[['module_presentation', 'threshold_days']],
    on='module_presentation',
    how='left'
)
studentVle_25pct = studentVle_25pct[studentVle_25pct['date'] <= studentVle_25pct['threshold_days']]

# Aggregate early behavior up to 25% completion
studentVle_25pct_agg = (
    studentVle_25pct
    .groupby('code_module_student_presentation_norm', as_index=False)
    .agg(
        early_total_clicks=('sum_click', 'sum'),
        early_avg_clicks=('sum_click', 'mean'),
        early_activity_count=('sum_click', 'count'),
        early_active_days=('date', 'nunique'),
        early_first_day=('date', 'min'),
        early_last_day=('date', 'max')
    )
)

# Merge 25%-completion features into the new dropout dataset
master_dropout_df = master_dropout_df.merge(
    studentVle_25pct_agg,
    left_on='code_module_student_presentation_norm',
    right_on='code_module_student_presentation_norm',
    how='left'
)

print('Adjusted master_dropout_df shape:', master_dropout_df.shape)
print('Dropout label value counts:')
print(master_dropout_df['dropout_flag'].value_counts(dropna=False))
print('\nEarly VLE feature sample:')
print(master_dropout_df[['code_module_student_presentation_norm', 'early_total_clicks', 'early_avg_clicks', 'early_activity_count']].head())
print('\nMissing rate for early VLE features:')
print(master_dropout_df[['early_total_clicks', 'early_avg_clicks', 'early_activity_count']].isna().mean().round(3))

Adjusted master_dropout_df shape: (173912, 26)
Dropout label value counts:
dropout_flag
0    160817
1     13095
Name: count, dtype: int64

Early VLE feature sample:
  code_module_student_presentation_norm  early_total_clicks  early_avg_clicks  \
0                       aaa-2013j-11391               278.0          7.128205   
1                       aaa-2013j-28400               308.0          3.710843   
2                       aaa-2013j-31604               191.0          4.152174   
3                       aaa-2013j-32885               342.0          3.931034   
4                       aaa-2013j-38053               375.0          4.573171   

   early_activity_count  
0                  39.0  
1                  83.0  
2                  46.0  
3                  87.0  
4                  82.0  

Missing rate for early VLE features:
early_total_clicks      0.197
early_avg_clicks        0.197
early_activity_count    0.197
dtype: float64


In [24]:
# Summary diagnostics for cleaned datasets
frames = {
    'assessment_df': assessment_df,
    'courses_cleaned_merged': courses_cleaned_merged,
    'student_data_cleaned': student_data_cleaned,
    'studentAssessment_df': studentAssessment_df,
    'studentVle': studentVle,
    'vle_df': vle_df,
    'student_registration_cleaned': df2 if 'df2' in globals() else None
}

for name, df in frames.items():
    if df is None:
        print(f"{name}: not loaded")
        continue
    print('='*80)
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print('Columns:', df.columns.tolist())
    print('Missing values:')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    print(missing[missing > 0].sort_values(ascending=False).head(20))
    print('Missing percent:')
    print(missing_pct[missing_pct > 0].sort_values(ascending=False).head(20))
    print('Dtypes:')
    print(df.dtypes.value_counts())
    if 'id_student' in df.columns or 'id_assessment' in df.columns:
        key_cols = [c for c in ['id_student','id_assessment','code_module_student_presentation','code_module_presentation'] if c in df.columns]
        for kc in key_cols:
            print(f"Unique {kc}: {df[kc].nunique()}/{len(df)}")
    if 'code_module_presentation' in df.columns and df['code_module_presentation'].dtype == object:
        print("Sample code_module_presentation unique values:", df['code_module_presentation'].unique()[:5])
    print()

Dataset: assessment_df
Shape: (206, 5)
Columns: ['id_assessment', 'assessment_type', 'date', 'weight', 'code_module_presentation']
Missing values:
Series([], dtype: int64)
Missing percent:
Series([], dtype: float64)
Dtypes:
object     2
float64    2
int64      1
Name: count, dtype: int64
Unique id_assessment: 206/206
Unique code_module_presentation: 22/206
Sample code_module_presentation unique values: ['aaa-2013j' 'aaa-2014j' 'bbb-2013b' 'bbb-2013j' 'bbb-2014b']

Dataset: courses_cleaned_merged
Shape: (22, 2)
Columns: ['module_presentation_length', 'code_module_presentation']
Missing values:
Series([], dtype: int64)
Missing percent:
Series([], dtype: float64)
Dtypes:
int64     1
object    1
Name: count, dtype: int64
Sample code_module_presentation unique values: ['aaa-2013j' 'aaa-2014j' 'bbb-2013j' 'bbb-2014j' 'bbb-2013b']

Dataset: student_data_cleaned
Shape: (32593, 13)
Columns: ['imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'code_module_presentati